In [ ]:

"""
T evaluation — timestep selection on a sample-level holdout.

Loads M checkpoint, wraps TimestepPredictor, and prepares true/predicted
m-grids for selection metrics (regret, gate, sparse-label budget sweep).
"""

import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "settings.py").exists():
    NOTEBOOK_DIR = NOTEBOOK_DIR / "models" / "modified-classification"
ROOT = NOTEBOOK_DIR.parents[1]
for p in (ROOT, NOTEBOOK_DIR):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

from model_m import MetricPredictor
from model_t import TimestepPredictor, ScalarStats, scalarize
from data_io import load_data, precompute_embeddings, df_to_metric_grids
from settings import *

def fig_title(subtitle: str) -> str:
    return f"{subtitle} for T on {METRICS_CSV.stem}, $\delta={{{TARGET_T_DELTA}}}$"

run_dirs = sorted(d for d in OUTPUTS_DIR.iterdir() if d.is_dir())
assert run_dirs, f"No run directories in {OUTPUTS_DIR}"
RUN_DIR = run_dirs[-1]
full_df = load_data()
all_sids = sorted(full_df.sample_id.unique())
rng = np.random.default_rng(SEED)
perm = rng.permutation(all_sids)
n_holdout = max(1, round(0.2 * len(perm)))
holdout_sids = sorted(perm[:n_holdout])
holdout_df = full_df[full_df.sample_id.isin(holdout_sids)].copy()
print(f"Run: {RUN_DIR.name}  holdout: {len(holdout_sids)} images  cells: {len(holdout_df)}")

T_START_VALUES = np.sort(holdout_df.t_start.unique())
T_END_VALUES = np.sort(holdout_df.t_end.unique())
N1, N2 = len(T_START_VALUES), len(T_END_VALUES)
CELLS = N1 * N2
SAMPLE_IDS = sorted(holdout_df.sample_id.unique())
N_IMG = len(SAMPLE_IDS)
SID_TO_K = {sid: k for k, sid in enumerate(SAMPLE_IDS)}
I_OF = {v: i for i, v in enumerate(T_START_VALUES)}
J_OF = {v: j for j, v in enumerate(T_END_VALUES)}
DEFAULT_I = int(np.argmin(np.abs(T_START_VALUES - DEFAULT_T_START)))
DEFAULT_J = int(np.argmin(np.abs(T_END_VALUES - DEFAULT_T_END)))

weights_path = RUN_DIR / "regressor_weights.pt"
ckpt = torch.load(weights_path, map_location="cpu", weights_only=False)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MetricPredictor(freeze_encoders=FREEZE_ENCODERS, device=DEVICE)
model.regressor.load_state_dict(ckpt["regressor_state_dict"])
model.regressor.set_target_stats(ckpt["target_mean"], ckpt["target_std"])
model.regressor.to(DEVICE).eval()
emb = precompute_embeddings(holdout_df.drop_duplicates("sample_id"), model, DEVICE)

true_psnr, _ = df_to_metric_grids(holdout_df, SAMPLE_IDS, T_START_VALUES, T_END_VALUES, "psnr")
true_clip, _ = df_to_metric_grids(holdout_df, SAMPLE_IDS, T_START_VALUES, T_END_VALUES, "clip")
SCALAR_STATS = ScalarStats(float(true_psnr.mean()), float(true_psnr.std()), float(true_clip.mean()), float(true_clip.std()))
true_m = scalarize(true_psnr, true_clip, SCALAR_STATS)

t_predictor = TimestepPredictor(model, t_start_values=T_START_VALUES, t_end_values=T_END_VALUES, scalar_stats=SCALAR_STATS)
pred_psnr = np.zeros_like(true_psnr)
pred_clip = np.zeros_like(true_clip)
pred_m = np.zeros_like(true_m)
for sid in SAMPLE_IDS:
    k = SID_TO_K[sid]
    grid = t_predictor.predict_grid_from_emb(emb[sid]["img"], emb[sid]["src"], emb[sid]["tar"])
    pred_psnr[k], pred_clip[k], pred_m[k] = grid.psnr_grid, grid.clip_grid, grid.m_grid

print(f"default cell ({DEFAULT_I}, {DEFAULT_J}) -> t_start={T_START_VALUES[DEFAULT_I]:.3g}, t_end={T_END_VALUES[DEFAULT_J]:.3g}")
print(f"holdout grids: {true_m.shape}")


In [ ]:

"""Per-image Spearman rho — can M rank cells within each image?"""

def per_image_spearman(true_grid, pred_grid):
    rhos = []
    for k in range(true_grid.shape[0]):
        rho, _ = spearmanr(true_grid[k].ravel(), pred_grid[k].ravel())
        rhos.append(np.nan if rho is None else float(rho))
    return np.array(rhos)

rho_psnr = per_image_spearman(true_psnr, pred_psnr)
rho_clip = per_image_spearman(true_clip, pred_clip)
rho_m = per_image_spearman(true_m, pred_m)
for name, r in [("PSNR", rho_psnr), ("CLIP", rho_clip), ("m", rho_m)]:
    rv = r[np.isfinite(r)]
    print(f"{name:5s}  median rho={np.nanmedian(r):.3f}  frac<0.2={np.mean(rv < 0.2):.2f}  frac<0={np.mean(rv < 0):.2f}")

fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ax, (name, r) in zip(axes, [("PSNR", rho_psnr), ("CLIP", rho_clip), ("m", rho_m)]):
    rv = r[np.isfinite(r)]
    if len(rv): ax.hist(rv, bins=min(30, max(3, len(rv))), edgecolor="white")
    ax.axvline(0, color="k", lw=0.5); ax.set_title(f"rho: {name}"); ax.set_xlabel("Spearman rho")
fig.suptitle(fig_title("Per-image Spearman"), fontsize=14); plt.tight_layout(); plt.show()


In [ ]:

"""Selection regret in m units (lower is better)."""

def regret(true_m_grid, pred_m_grid):
    out = np.zeros(true_m_grid.shape[0])
    for k in range(true_m_grid.shape[0]):
        chosen = np.unravel_index(pred_m_grid[k].argmax(), pred_m_grid[k].shape)
        out[k] = true_m_grid[k].max() - true_m_grid[k][chosen]
    return out

reg = regret(true_m, pred_m)
print(f"regret  median={np.median(reg):.4f}  p90={np.percentile(reg, 90):.4f}  frac~0={np.mean(reg < 1e-3):.2f}")
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(reg, bins=min(30, max(5, N_IMG // 2)), edgecolor="white")
ax.set_xlabel("regret (m units)"); ax.set_ylabel("count"); ax.set_title("selection regret")
fig.suptitle(fig_title("Selection regret"), fontsize=14); plt.tight_layout(); plt.show()


In [ ]:

"""Top-1 hit rate and top-3 overlap on m grid."""

def topk_metrics(true_m_grid, pred_m_grid, k=3):
    hit1, ov = [], []
    for i in range(true_m_grid.shape[0]):
        t_order = true_m_grid[i].ravel().argsort()[::-1]
        p_order = pred_m_grid[i].ravel().argsort()[::-1]
        hit1.append(t_order[0] == p_order[0])
        ov.append(len(set(t_order[:k]) & set(p_order[:k])) / k)
    return np.mean(hit1), np.mean(ov)

h1, ov3 = topk_metrics(true_m, pred_m, k=3)
print(f"top-1 hit rate={h1:.3f}   top-3 overlap={ov3:.3f}")


In [ ]:

"""Argmax-collapse: predicted vs true argmax spread across images."""

def argmax_cells(grid):
    return np.array([np.unravel_index(grid[k].argmax(), grid[k].shape) for k in range(grid.shape[0])])

true_arg = argmax_cells(true_m)
pred_arg = argmax_cells(pred_m)

def spread(arg):
    return arg[:, 0].std() + arg[:, 1].std()

print(f"true argmax spread = {spread(true_arg):.3f}")
print(f"pred argmax spread = {spread(pred_arg):.3f}")
print(f"unique pred cells = {len(set(map(tuple, pred_arg)))} / {CELLS}")

fig, axes = plt.subplots(1, 2, figsize=(8, 4), sharex=True, sharey=True)
for ax, arg, title in zip(axes, [true_arg, pred_arg], ["true argmax", "pred argmax"]):
    H = np.zeros((N1, N2))
    for i, j in arg: H[i, j] += 1
    ax.imshow(H, origin="lower", cmap="Blues"); ax.set_title(title)
    ax.scatter([DEFAULT_J], [DEFAULT_I], c="r", marker="x", label="default")
    ax.set_xlabel("t_end idx"); ax.set_ylabel("t_start idx"); ax.legend(fontsize=8)
fig.suptitle(fig_title("Argmax collapse"), fontsize=14); plt.tight_layout(); plt.show()


In [ ]:

"""Noise floor from second-seed re-renders (optional)."""

SECOND_SEED_CSV = None
if SECOND_SEED_CSV is None or not Path(SECOND_SEED_CSV).exists():
    print("SKIP: set SECOND_SEED_CSV to compute label noise floor.")
    noise_floor_m = np.nan
else:
    second = pd.read_csv(SECOND_SEED_CSV).rename(columns={PSNR_COL: "psnr", CLIP_COL: "clip"})
    if TARGET_T_DELTA is not None: second = second[second.t_delta == TARGET_T_DELTA]
    second = second[second.sample_id.isin(SAMPLE_IDS)]
    s_psnr, _ = df_to_metric_grids(second, SAMPLE_IDS, T_START_VALUES, T_END_VALUES, "psnr")
    s_clip, _ = df_to_metric_grids(second, SAMPLE_IDS, T_START_VALUES, T_END_VALUES, "clip")
    noise_floor_m = np.nanmedian(np.abs(true_m - scalarize(s_psnr, s_clip, SCALAR_STATS)))
    print(f"noise floor ~ {noise_floor_m:.4f}  vs regret median {np.median(reg):.4f}")


In [ ]:

"""Human-preferred cell rank under m (populate eyeball_cells)."""

eyeball_cells: dict[int, tuple[int, int]] = {}
if not eyeball_cells:
    print("SKIP: populate eyeball_cells with {image_index: (i, j)}.")
else:
    ranks = []
    for k, (i, j) in eyeball_cells.items():
        order = true_m[k].ravel().argsort()[::-1]
        rank = int(np.where(order == np.ravel_multi_index((i, j), (N1, N2)))[0][0])
        ranks.append(rank)
    print(f"median rank={np.median(ranks):.0f}/{CELLS}  frac top-5={np.mean(np.array(ranks) < 5):.2f}")


In [ ]:

"""Fresh-seed realized gain (requires render_and_score hook)."""

FRESH_SEED = None

def render_and_score(image_k, t_start, t_end, seed):
    raise NotImplementedError("Wire to ChordEditPipeline")

if FRESH_SEED is None:
    print("SKIP: set FRESH_SEED and implement render_and_score().")
else:
    realized = []
    for k in range(N_IMG):
        i, j = np.unravel_index(pred_m[k].argmax(), pred_m[k].shape)
        psnr_c, clip_c = render_and_score(k, T_START_VALUES[i], T_END_VALUES[j], FRESH_SEED)
        psnr_d, clip_d = render_and_score(k, T_START_VALUES[DEFAULT_I], T_END_VALUES[DEFAULT_J], FRESH_SEED)
        realized.append(scalarize(np.array([psnr_c]), np.array([clip_c]), SCALAR_STATS)[0]
                        - scalarize(np.array([psnr_d]), np.array([clip_d]), SCALAR_STATS)[0])
    realized = np.array(realized)
    nf = noise_floor_m if not np.isnan(noise_floor_m) else 0.0
    print(f"realized Δm median={np.median(realized):.4f}  frac>noise_floor={np.mean(realized > nf):.2f}")


In [ ]:

"""Deviate-or-default gate precision/recall."""

nf = noise_floor_m if "noise_floor_m" in dir() and not np.isnan(noise_floor_m) else NOISE_FLOOR_M
default_m = true_m[:, DEFAULT_I, DEFAULT_J]
truly_improvable = (true_m.max(axis=(1, 2)) - default_m) > nf
pred_gain = pred_m.max(axis=(1, 2)) - pred_m[:, DEFAULT_I, DEFAULT_J]
flagged = pred_gain > nf
tp = int(np.sum(flagged & truly_improvable))
precision = tp / max(int(flagged.sum()), 1)
recall = tp / max(int(truly_improvable.sum()), 1)
print(f"gate  precision={precision:.3f}  recall={recall:.3f}  ({flagged.sum()} flagged of {N_IMG})  noise_floor={nf:.4f}")


In [ ]:

"""Error stratification: regret vs default quality and grid-edge optima."""

df_err = pd.DataFrame({
    "sample_id": SAMPLE_IDS,
    "regret": reg,
    "default_quality": true_m[:, DEFAULT_I, DEFAULT_J],
    "best_at_edge": [(i in (0, N1-1)) or (j in (0, N2-1)) for i, j in true_arg],
})
print("regret vs default quality corr:", np.corrcoef(df_err["default_quality"], df_err["regret"])[0, 1])
print("regret by edge:", df_err.groupby("best_at_edge")["regret"].median().to_dict())
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].scatter(df_err["default_quality"], df_err["regret"], alpha=0.7)
axes[0].set_xlabel("default m"); axes[0].set_ylabel("regret")
axes[1].boxplot([df_err.loc[~df_err.best_at_edge, "regret"], df_err.loc[df_err.best_at_edge, "regret"]], tick_labels=["interior", "edge"])
axes[1].set_ylabel("regret")
fig.suptitle(fig_title("Error stratification"), fontsize=14); plt.tight_layout(); plt.show()


In [ ]:

"""Sparse labeling budget sweep — regret vs k."""

from active_label import cells_to_mask, policy_1d_sweeps, policy_structured_5x5, policy_default_only

def regret_on_mask(true_m_grid, pred_m_grid, mask):
    out = np.zeros(true_m_grid.shape[0])
    for k in range(true_m_grid.shape[0]):
        masked = np.where(mask[k], pred_m_grid[k], -np.inf)
        chosen = np.unravel_index(masked.argmax(), masked.shape)
        out[k] = true_m_grid[k].max() - true_m_grid[k][chosen]
    return out

def policy_mask(policy_name, k, rng):
    mask_3d = np.zeros((N_IMG, N1, N2), dtype=bool)
    if policy_name == "random":
        for ki in range(N_IMG):
            for idx in rng.choice(CELLS, size=min(k, CELLS), replace=False):
                i, j = np.unravel_index(idx, (N1, N2))
                mask_3d[ki, i, j] = True
        return mask_3d
    cells = policy_default_only() if policy_name == "default" else (
        policy_structured_5x5() if policy_name == "5x5" else policy_1d_sweeps())
    base = cells_to_mask(cells, T_START_VALUES, T_END_VALUES)
    return np.broadcast_to(base, (N_IMG, N1, N2)).copy()

budgets = [1, 9, 21, 25, 49, 121]
policies = ["default", "1d_sweeps", "5x5", "random"]
rng_budget = np.random.default_rng(SEED)
fig, ax = plt.subplots(figsize=(8, 5))
for policy_name in policies:
    xs, ys = [], []
    for k in budgets:
        if policy_name != "random" and k not in (1, 21, 25, 121): continue
        xs.append(k); ys.append(np.median(regret_on_mask(true_m, pred_m, policy_mask(policy_name, k, rng_budget))))
    ax.plot(xs, ys, marker="o", label=policy_name)
ax.axhline(np.median(reg), color="k", linestyle="--", alpha=0.5, label="dense argmax")
ax.set_xlabel("label budget k"); ax.set_ylabel("median regret"); ax.legend()
fig.suptitle(fig_title("Regret vs label budget"), fontsize=14); plt.tight_layout(); plt.show()


In [ ]:

"""T evaluation decision summary."""

print(fig_title("T summary"))
print(f"  holdout: {N_IMG} images  grid: {N1}x{N2}")
print(f"  Spearman m median: {np.nanmedian(rho_m):.3f}")
print(f"  regret median:     {np.median(reg):.4f}")
print(f"  top-1 / top-3:     {h1:.3f} / {ov3:.3f}")
print(f"  spread true/pred:  {spread(true_arg):.3f} / {spread(pred_arg):.3f}")
print(f"  gate prec/recall:  {precision:.3f} / {recall:.3f}")
